# 립리딩 전처리 · 학습 파이프라인 (Colab)

원본 영상을 Google Drive에 두고, 코랩 GPU로 전처리와 학습을 수행한다.

**실행 전 준비**
1. 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **GPU** 선택
2. Drive에 영상 폴더 생성 후 녹화본 업로드
3. 파일명 규칙: `{화자}_{문구}_{번호}.mp4` — 예) `s01_물주세요_01.mp4`

화자가 **2명 이상**이어야 학습이 진행된다. 화자 단위로 학습·검증을 나누기 때문이다.

## 1. 환경 확인

In [1]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("경고: 런타임 유형을 GPU로 변경하세요.")

torch 2.11.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Drive 마운트

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 3. 경로 설정

`DRIVE_ROOT` 아래 `raw/`에 영상을 두면, 전처리 결과가 `processed/`에 저장된다.
Drive에 저장하므로 런타임이 끊겨도 전처리를 다시 하지 않아도 된다.

In [3]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_RAW = DRIVE_ROOT / "raw"
DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"

for folder in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

videos = sorted(
    p.name for p in DRIVE_RAW.glob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov")
)
print(f"영상 {len(videos)}개")
for name in videos[:10]:
    print(f"  {name}")

영상 1077개
  s01_가래가있어요_01.mp4
  s01_가래가있어요_02.mp4
  s01_가래가있어요_03.mp4
  s01_가래가있어요_04.mp4
  s01_가래가있어요_05.mp4
  s01_가래가있어요_06.mp4
  s01_가래가있어요_07.mp4
  s01_가래가있어요_08.mp4
  s01_가래가있어요_09.mp4
  s01_가래가있어요_10.mp4


## 4. 저장소 clone

이미 받아둔 저장소가 있으면 `git pull`로 최신 코드만 가져온다.
clone이 중간에 실패해 빈 폴더가 남은 경우에는 지우고 다시 받는다.

비공개 저장소면 `https://<TOKEN>@github.com/...` 형태로 토큰을 넣는다.
토큰은 노트북에 저장하지 말고 매번 입력한다.

**코드를 갱신한 뒤에는 런타임을 다시 시작하거나 모듈을 reload해야 반영된다.**
파이썬은 한 번 import한 모듈을 다시 읽지 않는다.

```python
import importlib
import src.ml.training.dataset, src.ml.training.train
importlib.reload(src.ml.training.dataset)
importlib.reload(src.ml.training.train)
from src.ml.training.train import train
```

In [4]:
import os

REPO_URL = "https://github.com/HumanRhoid/hanium-lipreading.git"
BRANCH = "develop"
REPO_DIR = Path("/content/hanium-lipreading")

# clone이 중간에 실패하면 빈 폴더만 남아 다음 실행에서 git 명령이 어긋난다.
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !rm -rf {REPO_DIR}
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 경로: {Path.cwd()}")

Cloning into '/content/hanium-lipreading'...
remote: Enumerating objects: 608, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 608 (delta 71), reused 75 (delta 49), pack-reused 443 (from 1)
Receiving objects: 100% (608/608), 502.19 KiB | 10.04 MiB/s, done.
Resolving deltas: 100% (276/276), done.
작업 경로: /content/hanium-lipreading


## 5. 의존성 설치

`uv sync`는 쓰지 않는다. `pyproject.toml`이 torch를 **CPU 전용 인덱스**로 고정하고 있어
코랩의 GPU torch가 CPU 버전으로 교체되기 때문이다. 필요한 것만 pip로 설치한다.

In [5]:
!pip install --quiet mediapipe opencv-python wandb

import torch

print(f"설치 후 CUDA 사용 가능: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 13.9 MB/s eta 0:00:00
설치 후 CUDA 사용 가능: True


## 6. 얼굴 랜드마크 모델 내려받기

`face_landmarker.task`는 `.gitignore`에 제외돼 있어 저장소에 없다.

In [6]:
LANDMARKER_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
landmarker_path = REPO_DIR / "models" / "face_landmarker.task"
landmarker_path.parent.mkdir(parents=True, exist_ok=True)

if not landmarker_path.exists():
    !wget -q -O {landmarker_path} {LANDMARKER_URL}

print(f"{landmarker_path.name}: {landmarker_path.stat().st_size / 1e6:.1f} MB")

face_landmarker.task: 3.8 MB


## 7. 전처리 — 영상을 .npy로 변환

Drive를 입출력으로 직접 지정한다. 이미 변환된 파일은 건너뛴다.

In [7]:
import sys

sys.path.insert(0, str(REPO_DIR))

from src.ml.preprocess.vid2npy import run_batch

run_batch(raw_dir=DRIVE_RAW, processed_dir=DRIVE_PROCESSED)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_11.npy
이미 존재함, 건너뜀: /content

## 8. 매니페스트 생성

`.npy` 파일명을 파싱해 라벨과 화자를 뽑아낸다.

In [8]:
from scripts.build_manifest import build

manifest_path = DRIVE_ROOT / "manifest.csv"
build(processed_dir=DRIVE_PROCESSED, manifest_path=manifest_path)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest.csv
  클립 1077개 · 문구 15개 · 화자 7명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


In [9]:
import csv
from collections import Counter

with open(manifest_path, encoding="utf-8") as manifest_file:
    rows = list(csv.DictReader(manifest_file))

print(f"클립 {len(rows)}개")
print(f"화자별: {dict(Counter(row['speaker_id'] for row in rows))}")
print(f"문구별: {dict(Counter(row['label_text'] for row in rows))}")

클립 1077개
화자별: {'s01': 157, 's03': 148, 's04': 150, 's05': 150, 's06': 150, 's07': 171, 's08': 151}
문구별: {'가래가있어요': 70, '간호사불러주세요': 70, '더워요': 73, '도와주세요': 71, '물주세요': 77, '배고파요': 74, '보호자불러주세요': 71, '숨쉬기힘들어요': 72, '아파요': 70, '어지러워요': 71, '자세바꿔주세요': 72, '진통제주세요': 73, '추워요': 72, '토할거같아요': 71, '화장실가고싶어요': 70}


## 8-1. 학습 데이터를 로컬 디스크로 복사

Drive 마운트는 네트워크 파일시스템이라 매 에폭 수백 개를 원격에서 읽는다.
런타임 로컬 디스크로 옮기면 GPU가 데이터를 기다리는 시간이 줄어든다.

런타임이 끊기면 사라지므로 세션마다 다시 실행한다. 복사에 1~2분 걸린다.

In [10]:
import shutil
import time

LOCAL_ROOT = Path("/content/data")
LOCAL_PROCESSED = LOCAL_ROOT / "processed"

started = time.time()
if not LOCAL_PROCESSED.exists():
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# 매니페스트의 clip_path가 "processed/..." 라서 data_root만 바꾸면 그대로 맞는다.
TRAIN_ROOT = LOCAL_ROOT
local_count = len(list(LOCAL_PROCESSED.glob("*.npy")))
print(f"로컬 npy {local_count}개 · {time.time() - started:.0f}초")

로컬 npy 1077개 · 94초


## 9. 학습

체크포인트는 Drive에 저장되므로 런타임이 끊겨도 남는다.
학습 데이터는 `TRAIN_ROOT`(로컬 복사본)에서 읽어 I/O 대기를 줄인다.

**실험은 이 셀의 인자만 바꾸면 된다.** 저장소 코드를 고칠 필요가 없다.

- `seed` — 가중치 초기값·데이터 순서·증강을 한꺼번에 고정한다.
  같은 시드면 같은 결과가 나오므로 설정 비교의 전제가 된다
- `val_speakers` — 검증에 쓸 화자. **설정을 비교할 때는 반드시 고정한다.**
  `None`이면 화자 구성이 바뀔 때 검증 대상도 함께 바뀌어 비교가 깨진다
- `hidden_dim` / `num_layer` / `dropout` — 모델 크기와 정규화 강도
- `weight_decay` — 가중치를 작게 유지해 과적합을 억제
- `smoothing` — 최근 몇 에폭 평균으로 체크포인트를 판정할지. `1`이면 단일 에폭 최고치
- `label_smoothing` — 정답 확률을 100%로 몰지 않게 해 과신을 줄인다
- `grad_clip` — 드물게 튀는 그래디언트가 가중치를 흔드는 것을 막는다
- `ema_decay` — 가중치 이동평균으로 검증한다. 후반 진동이 완만해진다
- `augment` / `augmentation_config` — 증강 사용 여부와 강도
- `pretrained` — ImageNet 가중치로 백본을 초기화. 입력 정규화도 함께 바뀐다
- `freeze_backbone` — ResNet 층을 고정. `pretrained`와 함께 쓴다
- `amp` — bfloat16 혼합정밀도. GPU에서만 켜지고 속도가 2~3배 빨라진다
- `wandb_project` — 지정하면 실험이 웹 대시보드에 자동 기록된다

`label_smoothing` · `grad_clip` · `ema_decay`는 안정화 장치다. 셋 다 `0`을 주면
꺼진다. 체크포인트는 EMA를 켜면 평균 가중치로 저장되므로 검증 수치와 일치한다.

**시드 하나로 낸 결과는 그 자체로 성능이 아니다.** 같은 설정이라도 시드가 다르면
0.05~0.1 정도 흔들린다. 설정을 비교하거나 최종 수치를 낼 때는 시드 2~3개로
돌려 평균을 쓴다.

`pretrained=True`에 `freeze_backbone=False`면 학습률을 `3e-5`로 낮춘다. 좋은
초기값을 큰 보폭이 흐트러뜨리기 때문이다. 반대로 동결하면 움직이는 파라미터가
적어 `3e-4`까지 올려도 안정적이다.

**한 번에 하나만 바꾼다.** 두 개를 동시에 바꾸면 무엇이 효과였는지 알 수 없다.
`run_name`에 설정을 알아볼 수 있는 이름을 붙이면 나중에 비교하기 쉽다.

In [11]:
from src.ml.preprocess.augmentation import AugmentationConfig
from src.ml.training.train import train

# 증강 강도를 조절하려면 설정을 만들어 넘긴다. None이면 기본값을 쓴다.
strong_augmentation = AugmentationConfig(
    brightness_probability=0.7,
    contrast_probability=0.7,
    rotation_probability=0.6,
    shift_probability=0.6,
    zoom_probability=0.6,
)

best_accuracy = train(
    manifest_path=manifest_path,
    data_root=TRAIN_ROOT,
    epochs=80,
    batch_size=16,
    learning_rate=2e-4,
    seed=42,  # 가중치 초기값·데이터 순서·증강을 함께 고정한다
    val_speakers=["s04"],  # 설정 비교 시 고정. None이면 seed로 무작위 선택
    checkpoint_path=DRIVE_CHECKPOINTS / "best.pt",
    num_workers= 8,
    amp=True,
    hidden_dim=300,
    num_layer=2,
    dropout=0.3,
    weight_decay=0.01,
    smoothing=3,
    label_smoothing=0.1,  # 0이면 끔. 정답에 대한 과신을 줄인다
    grad_clip=1.0,  # 0이면 끔. 튀는 그래디언트를 잘라낸다
    ema_decay=0.998,  # 0이면 끔. 가중치 이동평균으로 검증한다
    augment=True,
    augmentation_config=None,  # strong_augmentation 으로 바꿔 강도 실험
    pretrained=False,  # True면 ImageNet 가중치 + ImageNet 입력 정규화
    freeze_backbone=False,  # pretrained와 함께 켜면 ResNet 층을 고정한다
    wandb_project="lipreading",
    run_name="baseline_s04",
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ssanta011205 (ssanta011205-seokyeong-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


장치: cuda | 클래스: 15개
학습 927개 · 검증 150개 클립 | 검증 화자 ['s04']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 4
label smoothing 0.1 · grad clip 1.0 · EMA 0.999 | 저장 기준 최근 3에폭 평균
[  1/60] train loss 2.7441 acc 0.076 | val loss 2.7094 acc 0.067 avg 0.067 | lr 9.99e-05
[  2/60] train loss 2.5312 acc 0.157 | val loss 2.7135 acc 0.067 avg 0.067 | lr 9.97e-05
[  3/60] train loss 2.2182 acc 0.309 | val loss 2.7216 acc 0.067 avg 0.067 | lr 9.94e-05
[  4/60] train loss 1.9286 acc 0.437 | val loss 2.7386 acc 0.067 avg 0.067 | lr 9.89e-05
[  5/60] train loss 1.6807 acc 0.590 | val loss 2.7728 acc 0.067 avg 0.067 | lr 9.83e-05
[  6/60] train loss 1.4685 acc 0.671 | val loss 2.8261 acc 0.067 avg 0.067 | lr 9.76e-05
[  7/60] train loss 1.2318 acc 0.754 | val loss 2.9068 acc 0.067 avg 0.067 | lr 9.67e-05
[  8/60] train loss 1.1099 acc 0.823 | val loss 3.0039 acc 0.067 avg 0.067 | lr 9.57e-05
[  9/60] train loss 1.0111 acc 0.851 | val loss 

lr,█████████▇▇▇▇▇▇▆▆▆▆▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
train/acc,▁▂▃▅▆███████████████████████████████████
train/loss,█▇▆▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇████████
val/acc_smoothed,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▇▇▇▇█████
val/loss,▅▅▅▅▆▆▇▇█████▇▇▇▇▇▇▆▅▅▅▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.58
best_val_acc_smoothed,0.58444
lr,0
train/acc,1
train/loss,0.56373


## 9-1. 교차검증

검증 화자 한 명으로 재면 그 사람의 난이도에 결과가 좌우된다. 화자를 바꿔가며
전부 한 번씩 검증으로 쓰고 평균을 내면 화자 편차에 흔들리지 않는 수치가 나온다.

**시드도 함께 반복해야 한다.** 같은 설정이라도 시드가 다르면 0.05~0.1 흔들리는데,
이 폭이 화자 간 차이와 비슷해서 한 번씩만 돌리면 순위를 신뢰할 수 없다.

`SEEDS`를 늘릴수록 신뢰도가 올라가지만 학습 횟수가 화자 수만큼 곱해진다.
경향만 볼 때는 시드 하나로, 발표에 쓸 최종 수치는 셋으로 돌린다.

In [ ]:
SEEDS = [42]  # [42, 1, 7] 로 늘리면 화자마다 여러 번 돌려 편차까지 본다

speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS}")

results = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'=' * 18} {speaker} · seed {seed} {'=' * 18}")
        results[(speaker, seed)] = train(
            manifest_path=manifest_path,
            data_root=TRAIN_ROOT,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv_{speaker}_seed{seed}",
        )

print(f"\n{'=' * 50}")
per_speaker = {}
for speaker in speakers:
    values = [results[(speaker, s)] for s in SEEDS]
    per_speaker[speaker] = sum(values) / len(values)
    detail = " ".join(f"{v:.3f}" for v in values)
    spread = f" (폭 {max(values) - min(values):.3f})" if len(values) > 1 else ""
    print(f"  {speaker}: {per_speaker[speaker]:.3f}   [{detail}]{spread}")

overall = sum(per_speaker.values()) / len(per_speaker)
gap = max(per_speaker.values()) - min(per_speaker.values())
print(f"\n전체 평균 {overall:.3f} · 화자 간 편차 {gap:.3f}")

화자 ['s01', 's03', 's04', 's05', 's06', 's07', 's08'] · 시드 [42]

================== s01 · seed 42 ==================


lr,█▇▆▅▃▁
train/acc,▁▂▃▅▇█
train/loss,█▇▅▄▂▁
val/acc,▁▁▁▁▁▁
val/acc_smoothed,▁▁▁▁▁▁
val/loss,▁▁▂▃▅█
lr,9e-05
train/acc,0.69579
train/loss,1.44145
val/acc,0.06667
val/acc_smoothed,0.06667


장치: cuda | 클래스: 15개
학습 920개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7524 acc 0.080 | val loss 2.7083 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/80] train loss 2.5415 acc 0.160 | val loss 2.7120 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/80] train loss 2.2442 acc 0.266 | val loss 2.7166 acc 0.064 avg 0.064 | lr 1.99e-04
[  4/80] train loss 1.9740 acc 0.421 | val loss 2.7282 acc 0.064 avg 0.064 | lr 1.99e-04
[  5/80] train loss 1.6330 acc 0.579 | val loss 2.7485 acc 0.064 avg 0.064 | lr 1.98e-04
[  6/80] train loss 1.4015 acc 0.696 | val loss 2.7977 acc 0.064 avg 0.064 | lr 1.97e-04
[  7/80] train loss 1.1984 acc 0.779 | val loss 2.8884 acc 0.064 avg 0.064 | lr 1.96e-04
[  8/80] train loss 1.0526 acc 0.837 | val loss 3.0026 acc 0.064 avg 0.064 | lr 1.95e-04
[  9/80] train loss 0.9637 acc 0.871 | val loss 

lr,███████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▂▄▅▆▇▇█████████████████████████████████
train/loss,█▇▆▆▄▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▁▂▂▃▄▄▆▆▇▇▇▇█▇▇▇████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▂▂▂▄▄▅▅▅▆▆▇▇█▇███████████████████
val/loss,▄▄▅▅▆██▇▆▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.57962
best_val_acc_smoothed,0.58386
lr,0
train/acc,1
train/loss,0.56271



================== s03 · seed 42 ==================


...


================== s07 · seed 42 ==================


장치: cuda | 클래스: 15개
학습 906개 · 검증 171개 클립 | 검증 화자 ['s07']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7388 acc 0.099 | val loss 2.7115 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/80] train loss 2.4893 acc 0.179 | val loss 2.7134 acc 0.070 avg 0.067 | lr 2.00e-04
[  3/80] train loss 2.1912 acc 0.315 | val loss 2.7171 acc 0.082 avg 0.072 | lr 1.99e-04
[  4/80] train loss 1.9324 acc 0.444 | val loss 2.7286 acc 0.082 avg 0.078 | lr 1.99e-04
[  5/80] train loss 1.6326 acc 0.594 | val loss 2.7454 acc 0.082 avg 0.082 | lr 1.98e-04
[  6/80] train loss 1.4412 acc 0.701 | val loss 2.7713 acc 0.082 avg 0.082 | lr 1.97e-04
[  7/80] train loss 1.2359 acc 0.772 | val loss 2.8220 acc 0.082 avg 0.082 | lr 1.96e-04
[  8/80] train loss 1.1137 acc 0.800 | val loss 2.8872 acc 0.082 avg 0.082 | lr 1.95e-04
[  9/80] train loss 0.9763 acc 0.874 | val loss 

lr,███████▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁
train/acc,▁▅▆▇████████████████████████████████████
train/loss,█▇▆▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▂▂▂▂▂▂▅▆▇█▇▆▆▆▅▅▅▅▅▆▅▅▄▅▅▅▅▅▅▅▅▅▅▅▄▄▅▅▅
val/acc_smoothed,▁▁▂▂▂▂▂▂▂▂▃▄▅▇▇█▆▆▇▇▇▆▆▆▆▆▆▅▆▆▆▆▆▆▅▅▅▅▆▆
val/loss,▁▁▁▂▃█▇▇▆▆▆▆▆▆▅▄▄▄▄▄▅▅▅▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂
best_val_acc,0.22222
best_val_acc_smoothed,0.21248
lr,0
train/acc,1
train/loss,0.56254



================== s08 · seed 42 ==================


장치: cuda | 클래스: 15개
학습 926개 · 검증 151개 클립 | 검증 화자 ['s08']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7787 acc 0.060 | val loss 2.7085 acc 0.073 avg 0.073 | lr 2.00e-04
[  2/80] train loss 2.6867 acc 0.075 | val loss 2.7077 acc 0.073 avg 0.073 | lr 2.00e-04
[  3/80] train loss 2.4385 acc 0.197 | val loss 2.7090 acc 0.066 avg 0.071 | lr 1.99e-04
[  4/80] train loss 2.2522 acc 0.246 | val loss 2.7117 acc 0.066 avg 0.068 | lr 1.99e-04
[  5/80] train loss 1.9717 acc 0.427 | val loss 2.7187 acc 0.066 avg 0.066 | lr 1.98e-04
[  6/80] train loss 1.7620 acc 0.528 | val loss 2.7345 acc 0.066 avg 0.066 | lr 1.97e-04
[  7/80] train loss 1.5214 acc 0.640 | val loss 2.7600 acc 0.066 avg 0.066 | lr 1.96e-04
[  8/80] train loss 1.2893 acc 0.738 | val loss 2.7926 acc 0.093 avg 0.075 | lr 1.95e-04
[  9/80] train loss 1.1357 acc 0.798 | val loss 

## 10. 체크포인트 확인

In [ ]:
checkpoint = torch.load(DRIVE_CHECKPOINTS / "best.pt", map_location="cpu")

print(f"에폭 {checkpoint['epoch']}")
print(f"클래스 {checkpoint['num_classes']}개")
print(f"검증 정확도 {checkpoint['val_accuracy']:.3f}")
print(f"최근 평균 {checkpoint['smoothed_accuracy']:.3f}")
print(
    f"모델 hidden {checkpoint['hidden_dim']} · "
    f"layer {checkpoint['num_layer']} · dropout {checkpoint['dropout']}"
)